# Metric Learning for Fine-Grained Car Retrieval

Pipeline complet pour retrieval fine-grained (96 classes marque+modèle).

**Composants :**
- Backbones : **ConvNeXt-Base** (CNN) et **DINOv2 ViT-B/14** (ViT)
- Loss : **SubCenter-ArcFace** (k=3 sous-centres)
- Sampling : **MPerClassSampler** (m=4)
- Résolutions adaptées RTX 5070 12 GB : **384×384** (ConvNeXt) / **224×224** (DINOv2)
- **Gradient accumulation** pour compenser les petits batches
- Augmentations fortes : RandAugment, ColorJitter, RandomErasing, RandomPerspective
- Mixed precision (AMP) + AdamW + Cosine LR schedule + warmup
- K-Fold 5 stratifié avec **checkpointing complet + reprise d'entraînement**
- **Weights & Biases** intégré (logging, artifacts, qualitative examples)
- **TTA** (original + flip horizontal)
- **k-reciprocal re-ranking** à l'évaluation
- **Ensemble** des deux backbones par concaténation L2-normalisée


## 1. Imports & configuration

In [ ]:
import os
import gc
import time
import math
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

from pytorch_metric_learning import losses, samplers

import wandb
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

SEED = 123
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# Réduit la fragmentation VRAM sur la 5070
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

load_dotenv()
WANDB_KEY = os.getenv("WANDB_API_KEY")
if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
    print("W&B: logged in")
else:
    print("W&B: pas de clé trouvée (set WANDB_API_KEY dans .env ou use_wandb=False)")

WANDB_PROJECT = "MIR_Cars_Project"

## 2. Chargement du dataset Cars (96 classes marque+modèle)

In [ ]:
data_dir = Path("../data/raw/Cars")

queries_to_exclude = {
    "0_1_BMW_X3_207.jpg", "0_0_BMW_Serie3Berline_74.jpg", "0_2_BMW_i8_299.jpg",
    "2_0_Volkswagen_Touareg_2822.jpg", "2_4_Volkswagen_Polo_3463.jpg", "2_9_Volkswagen_T-Roc_4209.jpg",
    "4_2_Opel_vivarofourgon_5999.jpg", "4_4_Opel_Insignatourer_6353.jpg", "4_9_Opel_zafiralife_6887.jpg",
    "6_0_Hyundai_Nexo_8282.jpg", "6_3_Hyundai_i10_8837.jpg", "6_5_Hyundai_i30_9125.jpg",
    "8_1_Ford_Puma_11276.jpg", "8_5_Ford_Explorer_11897.jpg", "8_6_Ford_Focus_11951.jpg",
}

all_image_paths = [f for f in data_dir.glob("*.jpg") if f.name not in queries_to_exclude]
all_labels = []
for img_path in all_image_paths:
    parts = img_path.stem.split('_')
    all_labels.append(f"{parts[0]}_{parts[1]}")

label_encoder = LabelEncoder()
all_labels_encoded = label_encoder.fit_transform(all_labels)
NUM_CLASSES = len(label_encoder.classes_)

print(f"Images gallery : {len(all_image_paths)}")
print(f"Classes uniques : {NUM_CLASSES}")

X = np.array(all_image_paths)
y = np.array(all_labels_encoded)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 3. Dataset & augmentations fortes

In [ ]:
class CarsDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img = Image.open(self.image_paths[index]).convert('RGB')
        label = int(self.labels[index])
        if self.transform:
            img = self.transform(img)
        return img, label


def build_transforms(img_size=384):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.10)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


## 4. Modèles — ConvNeXt-Base & DINOv2 ViT-B/14

Les deux backbones partagent une tête commune : projection linéaire vers `embedding_size=512`
suivie d'une L2-normalisation.


In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),
                            (x.size(-2), x.size(-1))).pow(1.0 / self.p)


class ConvNeXtMetricLearning(nn.Module):
    def __init__(self, embedding_size=512, pretrained=True, grad_checkpointing=False):
        super().__init__()
        self.backbone = timm.create_model(
            'convnext_base.fb_in22k_ft_in1k',
            pretrained=pretrained,
            num_classes=0,
            global_pool='',
        )
        if grad_checkpointing:
            self.backbone.set_grad_checkpointing(True)
        in_features = self.backbone.num_features  # 1024
        self.pool = GeM(p=3.0)
        self.bn = nn.BatchNorm1d(in_features)
        self.fc = nn.Linear(in_features, embedding_size)

    def forward(self, x):
        f = self.backbone.forward_features(x)
        f = self.pool(f).flatten(1)
        f = self.bn(f)
        f = self.fc(f)
        return F.normalize(f, p=2, dim=1)


class DINOv2MetricLearning(nn.Module):
    def __init__(self, embedding_size=512, pretrained=True, grad_checkpointing=False):
        super().__init__()
        self.backbone = timm.create_model(
            'vit_base_patch14_dinov2.lvd142m',
            pretrained=pretrained,
            num_classes=0,
            global_pool='token',
        )
        if grad_checkpointing:
            self.backbone.set_grad_checkpointing(True)
        in_features = self.backbone.num_features  # 768
        self.bn = nn.BatchNorm1d(in_features)
        self.fc = nn.Linear(in_features, embedding_size)

    def forward(self, x):
        f = self.backbone(x)
        f = self.bn(f)
        f = self.fc(f)
        return F.normalize(f, p=2, dim=1)


## 5. Entraînement générique avec SubCenter-ArcFace + W&B + Resume

- **Checkpointing complet** : model + criterion + optimizer + scheduler + scaler + epoch
- **Reprise d'entraînement** via `resume_from`
- **Sauvegarde `best` + `last`** (last utile si crash, SWA, etc.)
- **W&B** : config, train/val loss par epoch, qualitative examples, model artifact
- **Gradient accumulation** (`accum_steps`) pour compenser les petits batches sur la 5070

In [ ]:
def build_loaders(X_train, y_train, X_val, y_val, train_tf, val_tf,
                  batch_size=12, m_per_class=4, num_workers=2):
    train_ds = CarsDataset(X_train, y_train, transform=train_tf)
    val_ds   = CarsDataset(X_val,   y_val,   transform=val_tf)

    sampler = samplers.MPerClassSampler(
        labels=y_train, m=m_per_class,
        batch_size=batch_size,
        length_before_new_iter=len(X_train),
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=sampler,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    return train_loader, val_loader


def cosine_warmup_scheduler(optimizer, warmup_epochs, total_epochs, min_lr_ratio=0.01):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def save_checkpoint(path, *, model, criterion, optimizer, scheduler, scaler,
                    epoch, val_loss, train_loss, num_classes, embedding_size, sub_centers):
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'criterion_state': criterion.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict() if scaler is not None else None,
        'val_loss': val_loss,
        'train_loss': train_loss,
        'num_classes': num_classes,
        'embedding_size': embedding_size,
        'sub_centers': sub_centers,
    }, path)


def train_metric_learning(
    model,
    train_loader,
    val_loader,
    num_classes,
    embedding_size=512,
    num_epochs=60,
    lr=1e-4,
    weight_decay=1e-4,
    warmup_epochs=3,
    patience=12,
    save_path="models/best.pth",
    sub_centers=3,
    accum_steps=1,
    resume_from=None,
    # W&B
    use_wandb=True,
    wandb_project=WANDB_PROJECT,
    wandb_run_name=None,
    wandb_config_extra=None,
):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    last_path = save_path.replace('.pth', '_last.pth')
    model = model.to(device)

    criterion = losses.SubCenterArcFaceLoss(
        num_classes=num_classes,
        embedding_size=embedding_size,
        margin=28.6,
        scale=64,
        sub_centers=sub_centers,
    ).to(device)

    optimizer = optim.AdamW(
        [
            {'params': model.parameters()},
            {'params': criterion.parameters()},
        ],
        lr=lr, weight_decay=weight_decay,
    )
    scheduler = cosine_warmup_scheduler(optimizer, warmup_epochs, num_epochs)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    start_epoch = 0
    best_val_loss = float('inf')
    if resume_from is not None and os.path.exists(resume_from):
        print(f"Resuming from {resume_from}")
        ckpt = torch.load(resume_from, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        criterion.load_state_dict(ckpt['criterion_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        if scaler is not None and ckpt.get('scaler_state') is not None:
            scaler.load_state_dict(ckpt['scaler_state'])
        start_epoch = ckpt['epoch']
        best_val_loss = ckpt['val_loss']
        print(f"  resumed at epoch {start_epoch}, best val_loss so far: {best_val_loss:.4f}")

    run = None
    if use_wandb:
        config = {
            "architecture": type(model).__name__,
            "num_classes": num_classes,
            "embedding_size": embedding_size,
            "epochs": num_epochs,
            "batch_size": train_loader.batch_size,
            "accum_steps": accum_steps,
            "effective_batch_size": train_loader.batch_size * accum_steps,
            "lr": lr,
            "weight_decay": weight_decay,
            "warmup_epochs": warmup_epochs,
            "patience": patience,
            "loss": f"SubCenterArcFace(k={sub_centers}, margin=28.6, scale=64)",
            "optimizer": "AdamW",
            "scheduler": "CosineWarmup",
            "amp": scaler is not None,
            "resumed_from": resume_from,
            "start_epoch": start_epoch,
        }
        if wandb_config_extra:
            config.update(wandb_config_extra)
        run = wandb.init(
            project=wandb_project,
            name=wandb_run_name,
            config=config,
            reinit=True,
        )
        wandb.watch(model, log="gradients", log_freq=200)

    epochs_no_improve = 0
    print(f"\nDébut de l'entraînement sur {device}")
    print(f"  save_path: {save_path}")
    print(f"  batch_size={train_loader.batch_size} × accum_steps={accum_steps} = effective {train_loader.batch_size * accum_steps}")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        t0 = time.time()
        run_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=scaler is not None):
                emb = model(imgs)
                loss = criterion(emb, labels) / accum_steps

            if scaler is not None:
                scaler.scale(loss).backward()
                if (step + 1) % accum_steps == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            else:
                loss.backward()
                if (step + 1) % accum_steps == 0:
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

            run_loss += loss.item() * accum_steps

        avg_train = run_loss / max(1, len(train_loader))

        model.eval()
        run_val = 0.0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=scaler is not None):
                    emb = model(imgs)
                    loss = criterion(emb, labels)
                run_val += loss.item()
        avg_val = run_val / max(1, len(val_loader))

        scheduler.step()
        dt = time.time() - t0
        current_lr = optimizer.param_groups[0]['lr']
        gpu_mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == 'cuda' else 0
        print(f"Epoch {epoch+1:3d}/{num_epochs} | train {avg_train:.4f} | val {avg_val:.4f} | lr {current_lr:.2e} | {dt:.1f}s | peak VRAM {gpu_mem:.1f} GB")

        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": avg_train,
                "val/loss": avg_val,
                "lr": current_lr,
                "epoch_time_s": dt,
                "peak_vram_gb": gpu_mem,
                "best_val_loss_so_far": min(best_val_loss, avg_val),
            }, step=epoch + 1)
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()

        save_checkpoint(
            last_path,
            model=model, criterion=criterion, optimizer=optimizer,
            scheduler=scheduler, scaler=scaler,
            epoch=epoch + 1, val_loss=avg_val, train_loss=avg_train,
            num_classes=num_classes, embedding_size=embedding_size,
            sub_centers=sub_centers,
        )

        if avg_val < best_val_loss - 1e-4:
            best_val_loss = avg_val
            epochs_no_improve = 0
            save_checkpoint(
                save_path,
                model=model, criterion=criterion, optimizer=optimizer,
                scheduler=scheduler, scaler=scaler,
                epoch=epoch + 1, val_loss=avg_val, train_loss=avg_train,
                num_classes=num_classes, embedding_size=embedding_size,
                sub_centers=sub_centers,
            )
            print(f"  >>> best checkpoint saved (val_loss={avg_val:.4f})")
            if use_wandb:
                wandb.run.summary["best_val_loss"] = best_val_loss
                wandb.run.summary["best_epoch"] = epoch + 1
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    if use_wandb and run is not None:
        try:
            artifact = wandb.Artifact(
                name=f"best_model_{wandb_run_name or 'run'}".replace('/', '_'),
                type="model",
                metadata={"val_loss": best_val_loss},
            )
            artifact.add_file(save_path)
            run.log_artifact(artifact)
        except Exception as e:
            print(f"W&B artifact upload skipped: {e}")
        run.finish()

    return best_val_loss


## 6. Boucle K-Fold générique

In [ ]:
def run_kfold(
    backbone_name,
    model_factory,
    img_size,
    batch_size,
    num_epochs,
    accum_steps=1,
    lr=1e-4,
    save_dir="models",
    num_workers=2,
    use_wandb=True,
    resume=False,
):
    os.makedirs(save_dir, exist_ok=True)
    train_tf, val_tf = build_transforms(img_size=img_size)

    global_best_loss = float('inf')
    global_best_path = os.path.join(save_dir, f"best_GLOBAL_{backbone_name}.pth")
    best_fold = -1

    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n{'='*60}\n  {backbone_name} — FOLD {fold_idx}/5\n{'='*60}")
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_va, y_va = X[va_idx], y[va_idx]

        train_loader, val_loader = build_loaders(
            X_tr, y_tr, X_va, y_va, train_tf, val_tf,
            batch_size=batch_size, m_per_class=4, num_workers=num_workers,
        )

        model = model_factory()
        fold_path = os.path.join(save_dir, f"best_{backbone_name}_fold{fold_idx}.pth")
        last_path = fold_path.replace('.pth', '_last.pth')

        resume_from = last_path if (resume and os.path.exists(last_path)) else None

        best_loss = train_metric_learning(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_classes=NUM_CLASSES,
            embedding_size=512,
            num_epochs=num_epochs,
            lr=lr,
            save_path=fold_path,
            accum_steps=accum_steps,
            resume_from=resume_from,
            use_wandb=use_wandb,
            wandb_run_name=f"{backbone_name}_fold{fold_idx}",
            wandb_config_extra={"fold": fold_idx, "backbone": backbone_name,
                                "img_size": img_size},
        )

        if best_loss < global_best_loss:
            global_best_loss = best_loss
            best_fold = fold_idx
            shutil.copy(fold_path, global_best_path)
            print(f"  >>> New global best ({backbone_name}): fold {fold_idx} val_loss={best_loss:.4f}")

        del model, train_loader, val_loader
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    print(f"\n{backbone_name} — Best fold: {best_fold} | Best val_loss: {global_best_loss:.4f}")
    return global_best_path


## 7. Entraînement ConvNeXt-Base (RTX 5070 12 GB)

In [ ]:
# RTX 5070 12 GB — réglages validés :
#   img_size=384, batch_size=12, accum_steps=2  => batch effectif 24
# Si OOM : passe à ConvNeXtMetricLearning(grad_checkpointing=True)
#          ou img_size=320, ou batch_size=8 + accum_steps=3
convnext_path = run_kfold(
    backbone_name="ConvNeXtBase",
    model_factory=lambda: ConvNeXtMetricLearning(embedding_size=512, grad_checkpointing=False),
    img_size=384,
    batch_size=12,
    accum_steps=2,
    num_epochs=60,
    lr=1e-4,
    num_workers=2,
    use_wandb=True,
    resume=False,     # passe à True pour reprendre automatiquement depuis _last
)


## 8. Entraînement DINOv2 ViT-B/14 (RTX 5070 12 GB)

In [ ]:
# DINOv2 : patch=14, résolutions valides multiples de 14 (196, 224, 392, 448...).
# RTX 5070 : 224 avec batch_size=24, accum_steps=2 => batch effectif 48.
dinov2_path = run_kfold(
    backbone_name="DINOv2_ViTB14",
    model_factory=lambda: DINOv2MetricLearning(embedding_size=512, grad_checkpointing=False),
    img_size=224,
    batch_size=24,
    accum_steps=2,
    num_epochs=50,
    lr=5e-5,
    num_workers=2,
    use_wandb=True,
    resume=False,
)


## 9. Chargement d'un checkpoint (sans relancer l'entraînement)

Utile si tu veux juste évaluer à partir d'un modèle déjà entraîné, ou reprendre
après un crash.

In [ ]:
def load_checkpoint(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    model = model.to(device).eval()
    print(f"Loaded {ckpt_path} (epoch {ckpt.get('epoch')}, val_loss {ckpt.get('val_loss'):.4f})")
    return model


# Si tu relances la notebook sans ré-entraîner, décommente et adapte :
# convnext_path = "models/best_GLOBAL_ConvNeXtBase.pth"
# dinov2_path   = "models/best_GLOBAL_DINOv2_ViTB14.pth"


## 10. Extraction des embeddings + TTA (flip horizontal)

In [ ]:
@torch.no_grad()
def extract_embeddings(model, dataloader, use_tta=True):
    model.eval()
    feats, labs = [], []
    for imgs, labels in dataloader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=device.type == 'cuda'):
            e1 = model(imgs)
            if use_tta:
                e2 = model(torch.flip(imgs, dims=[3]))
                e = F.normalize(e1 + e2, p=2, dim=1)
            else:
                e = e1
        feats.append(e.float().cpu().numpy())
        labs.append(labels.numpy() if torch.is_tensor(labels) else np.asarray(labels))
    return np.concatenate(feats), np.concatenate(labs)


## 11. Préparation gallery & queries

In [ ]:
# Queries officielles — groupes impairs. Adapte selon ton groupe.
query_filenames = [
    "0_1_BMW_X3_207.jpg", "0_0_BMW_Serie3Berline_74.jpg", "0_2_BMW_i8_299.jpg",
    "2_0_Volkswagen_Touareg_2822.jpg", "2_4_Volkswagen_Polo_3463.jpg", "2_9_Volkswagen_T-Roc_4209.jpg",
    "4_2_Opel_vivarofourgon_5999.jpg", "4_4_Opel_Insignatourer_6353.jpg", "4_9_Opel_zafiralife_6887.jpg",
    "6_0_Hyundai_Nexo_8282.jpg", "6_3_Hyundai_i10_8837.jpg", "6_5_Hyundai_i30_9125.jpg",
    "8_1_Ford_Puma_11276.jpg", "8_5_Ford_Explorer_11897.jpg", "8_6_Ford_Focus_11951.jpg",
]

query_paths = [data_dir / q for q in query_filenames]
query_labels = []
for q in query_filenames:
    parts = q.split('_')
    q_label_str = f"{parts[0]}_{parts[1]}"   # marque_modèle (96 classes)
    query_labels.append(label_encoder.transform([q_label_str])[0])
query_labels = np.array(query_labels)

print(f"Queries : {len(query_paths)}")
print(f"Gallery : {len(X)}")


def build_eval_loaders(img_size, batch_size=16, num_workers=2):
    _, val_tf = build_transforms(img_size=img_size)
    gallery_ds = CarsDataset(X, y, transform=val_tf)
    query_ds   = CarsDataset(np.array(query_paths), query_labels, transform=val_tf)
    gallery_loader = DataLoader(gallery_ds, batch_size=batch_size, shuffle=False,
                                num_workers=num_workers, pin_memory=True)
    query_loader   = DataLoader(query_ds, batch_size=batch_size, shuffle=False,
                                num_workers=num_workers, pin_memory=True)
    return gallery_loader, query_loader


## 12. Extraction des features pour chaque backbone

In [ ]:
conv_model = ConvNeXtMetricLearning(embedding_size=512)
conv_model = load_checkpoint(conv_model, convnext_path)
conv_gallery_loader, conv_query_loader = build_eval_loaders(img_size=384, batch_size=16)

t0 = time.time()
conv_gallery_feats, conv_gallery_labels = extract_embeddings(conv_model, conv_gallery_loader, use_tta=True)
print(f"ConvNeXt gallery : {conv_gallery_feats.shape} in {time.time()-t0:.1f}s")

t0 = time.time()
conv_query_feats, conv_query_labels = extract_embeddings(conv_model, conv_query_loader, use_tta=True)
print(f"ConvNeXt query   : {conv_query_feats.shape} in {time.time()-t0:.1f}s")

del conv_model; gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()


In [ ]:
dino_model = DINOv2MetricLearning(embedding_size=512)
dino_model = load_checkpoint(dino_model, dinov2_path)
dino_gallery_loader, dino_query_loader = build_eval_loaders(img_size=224, batch_size=32)

t0 = time.time()
dino_gallery_feats, dino_gallery_labels = extract_embeddings(dino_model, dino_gallery_loader, use_tta=True)
print(f"DINOv2 gallery : {dino_gallery_feats.shape} in {time.time()-t0:.1f}s")

t0 = time.time()
dino_query_feats, dino_query_labels = extract_embeddings(dino_model, dino_query_loader, use_tta=True)
print(f"DINOv2 query   : {dino_query_feats.shape} in {time.time()-t0:.1f}s")

del dino_model; gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()

assert np.array_equal(conv_gallery_labels, dino_gallery_labels)
assert np.array_equal(conv_query_labels, dino_query_labels)
gallery_labels = conv_gallery_labels
q_labels = conv_query_labels


## 13. Métriques : Recall, Precision, AP, mAP, R-Precision

In [ ]:
def compute_similarity(query_feats, gallery_feats):
    return query_feats @ gallery_feats.T


def average_precision(retrieved_labels, true_label, k=None):
    if k is not None:
        retrieved_labels = retrieved_labels[:k]
    hits = (retrieved_labels == true_label).astype(np.float32)
    if hits.sum() == 0:
        return 0.0
    precisions = np.cumsum(hits) / (np.arange(len(hits)) + 1)
    return float((precisions * hits).sum() / hits.sum())


def evaluate_retrieval(query_feats, gallery_feats, q_labels, g_labels,
                       top_ks=(20, 50, 100), verbose=True):
    sim = compute_similarity(query_feats, gallery_feats)
    order = np.argsort(-sim, axis=1)
    n_queries = len(q_labels)

    per_query = []
    for i in range(n_queries):
        row = {'query_idx': i, 'true_label': int(q_labels[i])}
        ranked_labels = g_labels[order[i]]
        n_relevant = int((g_labels == q_labels[i]).sum())
        row['n_relevant'] = n_relevant
        for k in top_ks:
            topk = ranked_labels[:k]
            hits = (topk == q_labels[i]).astype(np.float32)
            tp = int(hits.sum())
            row[f'R@{k}']  = tp / max(1, n_relevant)
            row[f'P@{k}']  = tp / k
            row[f'AP@{k}'] = average_precision(ranked_labels, q_labels[i], k=k)
        R = max(1, n_relevant)
        row['R-Prec'] = float((ranked_labels[:R] == q_labels[i]).sum() / R)
        per_query.append(row)

    df = pd.DataFrame(per_query)
    summary = {}
    for k in top_ks:
        summary[f'mR@{k}']   = df[f'R@{k}'].mean()
        summary[f'mP@{k}']   = df[f'P@{k}'].mean()
        summary[f'mAP@{k}']  = df[f'AP@{k}'].mean()
    summary['mR-Prec'] = df['R-Prec'].mean()

    if verbose:
        print("--- Summary ---")
        for k, v in summary.items():
            print(f"  {k:10s}: {v*100:6.2f}%")
    return df, summary, sim, order


## 14. Évaluation baseline — ConvNeXt, DINOv2, Ensemble

In [ ]:
print("=== ConvNeXt-Base (TTA) ===")
df_conv, sum_conv, sim_conv, _ = evaluate_retrieval(
    conv_query_feats, conv_gallery_feats, q_labels, gallery_labels)

print("\n=== DINOv2 ViT-B/14 (TTA) ===")
df_dino, sum_dino, sim_dino, _ = evaluate_retrieval(
    dino_query_feats, dino_gallery_feats, q_labels, gallery_labels)


In [ ]:
def concat_and_normalize(a, b, w_a=1.0, w_b=1.0):
    a = a * w_a
    b = b * w_b
    x = np.concatenate([a, b], axis=1)
    x = x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)
    return x

ens_gallery = concat_and_normalize(conv_gallery_feats, dino_gallery_feats)
ens_query   = concat_and_normalize(conv_query_feats,   dino_query_feats)

print("=== Ensemble ConvNeXt + DINOv2 (TTA) ===")
df_ens, sum_ens, sim_ens, _ = evaluate_retrieval(
    ens_query, ens_gallery, q_labels, gallery_labels)


## 15. K-Reciprocal Re-Ranking (Zhong et al., CVPR 2017)

In [ ]:
def k_reciprocal_neigh(initial_rank, i, k1):
    forward_k = initial_rank[i, :k1 + 1]
    backward_k = initial_rank[forward_k, :k1 + 1]
    fi = np.where(backward_k == i)[0]
    return forward_k[fi]


def re_ranking(query_feats, gallery_feats, k1=20, k2=6, lambda_value=0.3):
    query_num = query_feats.shape[0]
    all_feats = np.concatenate([query_feats, gallery_feats], axis=0).astype(np.float32)
    all_num = all_feats.shape[0]

    original_dist = 1.0 - all_feats @ all_feats.T
    original_dist = np.maximum(original_dist, 0.0)
    original_dist = np.transpose(original_dist / (np.max(original_dist, axis=0) + 1e-12))

    V = np.zeros_like(original_dist, dtype=np.float16)
    initial_rank = np.argsort(original_dist, axis=1).astype(np.int32)

    for i in range(all_num):
        k_reciprocal_index = k_reciprocal_neigh(initial_rank, i, k1)
        k_reciprocal_expansion_index = k_reciprocal_index
        for j, candidate in enumerate(k_reciprocal_index):
            candidate_k_reciprocal_index = k_reciprocal_neigh(initial_rank, candidate, int(np.around(k1/2)))
            if len(np.intersect1d(candidate_k_reciprocal_index, k_reciprocal_index)) \
                    > 2./3 * len(candidate_k_reciprocal_index):
                k_reciprocal_expansion_index = np.append(k_reciprocal_expansion_index,
                                                         candidate_k_reciprocal_index)
        k_reciprocal_expansion_index = np.unique(k_reciprocal_expansion_index)
        weight = np.exp(-original_dist[i, k_reciprocal_expansion_index])
        V[i, k_reciprocal_expansion_index] = (weight / np.sum(weight)).astype(np.float16)

    original_dist = original_dist[:query_num, :]

    if k2 != 1:
        V_qe = np.zeros_like(V, dtype=np.float16)
        for i in range(all_num):
            V_qe[i, :] = np.mean(V[initial_rank[i, :k2], :], axis=0)
        V = V_qe
        del V_qe

    invIndex = [np.where(V[:, i] != 0)[0] for i in range(all_num)]

    jaccard_dist = np.zeros_like(original_dist, dtype=np.float16)
    for i in range(query_num):
        temp_min = np.zeros(shape=[1, all_num], dtype=np.float16)
        indNonZero = np.where(V[i, :] != 0)[0]
        indImages = [invIndex[ind] for ind in indNonZero]
        for j in range(len(indNonZero)):
            temp_min[0, indImages[j]] += np.minimum(V[i, indNonZero[j]], V[indImages[j], indNonZero[j]])
        jaccard_dist[i] = 1 - temp_min / (2 - temp_min)

    final_dist = jaccard_dist * (1 - lambda_value) + original_dist * lambda_value
    final_dist = final_dist[:query_num, query_num:]
    return final_dist


def evaluate_from_distance(dist, q_labels, g_labels, top_ks=(20, 50, 100), verbose=True):
    order = np.argsort(dist, axis=1)
    n_queries = len(q_labels)

    per_query = []
    for i in range(n_queries):
        row = {'query_idx': i, 'true_label': int(q_labels[i])}
        ranked_labels = g_labels[order[i]]
        n_relevant = int((g_labels == q_labels[i]).sum())
        row['n_relevant'] = n_relevant
        for k in top_ks:
            topk = ranked_labels[:k]
            hits = (topk == q_labels[i]).astype(np.float32)
            tp = int(hits.sum())
            row[f'R@{k}']  = tp / max(1, n_relevant)
            row[f'P@{k}']  = tp / k
            row[f'AP@{k}'] = average_precision(ranked_labels, q_labels[i], k=k)
        R = max(1, n_relevant)
        row['R-Prec'] = float((ranked_labels[:R] == q_labels[i]).sum() / R)
        per_query.append(row)

    df = pd.DataFrame(per_query)
    summary = {}
    for k in top_ks:
        summary[f'mR@{k}']  = df[f'R@{k}'].mean()
        summary[f'mP@{k}']  = df[f'P@{k}'].mean()
        summary[f'mAP@{k}'] = df[f'AP@{k}'].mean()
    summary['mR-Prec'] = df['R-Prec'].mean()
    if verbose:
        print("--- Summary (re-ranked) ---")
        for k, v in summary.items():
            print(f"  {k:10s}: {v*100:6.2f}%")
    return df, summary, order


## 16. Évaluation finale avec re-ranking

In [ ]:
print("=== ConvNeXt + k-reciprocal re-ranking ===")
dist_conv_rr = re_ranking(conv_query_feats, conv_gallery_feats, k1=20, k2=6, lambda_value=0.3)
df_conv_rr, sum_conv_rr, _ = evaluate_from_distance(dist_conv_rr, q_labels, gallery_labels)

print("\n=== DINOv2 + k-reciprocal re-ranking ===")
dist_dino_rr = re_ranking(dino_query_feats, dino_gallery_feats, k1=20, k2=6, lambda_value=0.3)
df_dino_rr, sum_dino_rr, _ = evaluate_from_distance(dist_dino_rr, q_labels, gallery_labels)

print("\n=== Ensemble + k-reciprocal re-ranking ===")
dist_ens_rr = re_ranking(ens_query, ens_gallery, k1=20, k2=6, lambda_value=0.3)
df_ens_rr, sum_ens_rr, order_ens_rr = evaluate_from_distance(dist_ens_rr, q_labels, gallery_labels)


## 17. Tableau comparatif final + log W&B

In [ ]:
def summary_row(name, s):
    return {
        'Method': name,
        'mAP@20': f"{s['mAP@20']*100:.2f}%",
        'mAP@50': f"{s['mAP@50']*100:.2f}%",
        'mAP@100': f"{s['mAP@100']*100:.2f}%",
        'mR@50': f"{s['mR@50']*100:.2f}%",
        'mR@100': f"{s['mR@100']*100:.2f}%",
        'R-Prec': f"{s['mR-Prec']*100:.2f}%",
    }

results = pd.DataFrame([
    summary_row("ConvNeXt-Base (TTA)",           sum_conv),
    summary_row("ConvNeXt-Base + reranking",     sum_conv_rr),
    summary_row("DINOv2 ViT-B/14 (TTA)",         sum_dino),
    summary_row("DINOv2 ViT-B/14 + reranking",   sum_dino_rr),
    summary_row("Ensemble (TTA)",                sum_ens),
    summary_row("Ensemble + reranking",          sum_ens_rr),
])
print(results.to_string(index=False))

if WANDB_KEY:
    eval_run = wandb.init(project=WANDB_PROJECT, name="final_evaluation", reinit=True)
    for name, s in [
        ("ConvNeXt",       sum_conv),
        ("ConvNeXt_RR",    sum_conv_rr),
        ("DINOv2",         sum_dino),
        ("DINOv2_RR",      sum_dino_rr),
        ("Ensemble",       sum_ens),
        ("Ensemble_RR",    sum_ens_rr),
    ]:
        wandb.log({f"{name}/{k}": v for k, v in s.items()})
    wandb.log({"final_results": wandb.Table(dataframe=results)})
    eval_run.finish()


## 18. Visualisation des top-k pour chaque query + log W&B

In [ ]:
def show_top_results_from_order(order, query_idx, top_k=10, title_prefix="Ensemble+RR"):
    q_img_path = query_paths[query_idx]
    q_label = q_labels[query_idx]
    q_label_name = label_encoder.inverse_transform([q_label])[0]

    top_indices = order[query_idx, :top_k]

    fig, axes = plt.subplots(1, top_k + 1, figsize=(2.2*(top_k+1), 3))
    axes[0].imshow(Image.open(q_img_path))
    axes[0].set_title(f"QUERY\n{q_label_name}", color='blue', fontsize=9)
    axes[0].axis('off')

    for i, idx in enumerate(top_indices):
        res_path = X[idx]
        res_label = gallery_labels[idx]
        res_label_name = label_encoder.inverse_transform([res_label])[0]
        axes[i+1].imshow(Image.open(res_path))
        color = 'green' if res_label == q_label else 'red'
        axes[i+1].set_title(f"Top {i+1}\n{res_label_name}", color=color, fontsize=8)
        axes[i+1].axis('off')

    plt.suptitle(f"{title_prefix} — Query {query_idx+1}", fontsize=11)
    plt.tight_layout()
    return fig


for qi in range(len(query_paths)):
    fig = show_top_results_from_order(order_ens_rr, qi, top_k=10)
    plt.show()


if WANDB_KEY:
    qual_run = wandb.init(project=WANDB_PROJECT, name="qualitative_examples", reinit=True)
    for qi in range(len(query_paths)):
        q_label_name = label_encoder.inverse_transform([q_labels[qi]])[0]
        imgs = [wandb.Image(str(query_paths[qi]), caption=f"QUERY: {q_label_name}")]
        for idx in order_ens_rr[qi, :5]:
            res_label_name = label_encoder.inverse_transform([gallery_labels[idx]])[0]
            correct = "OK" if gallery_labels[idx] == q_labels[qi] else "KO"
            imgs.append(wandb.Image(str(X[idx]), caption=f"[{correct}] {res_label_name}"))
        wandb.log({f"query_{qi+1}_{q_label_name}": imgs})
    qual_run.finish()


## 19. Sauvegarde des features (`.npy`) pour la Part III

In [ ]:
os.makedirs("features", exist_ok=True)

np.save("features/conv_gallery.npy", conv_gallery_feats)
np.save("features/conv_query.npy",   conv_query_feats)
np.save("features/dino_gallery.npy", dino_gallery_feats)
np.save("features/dino_query.npy",   dino_query_feats)
np.save("features/ens_gallery.npy",  ens_gallery)
np.save("features/ens_query.npy",    ens_query)
np.save("features/gallery_labels.npy", gallery_labels)
np.save("features/query_labels.npy",   q_labels)

for f in sorted(os.listdir("features")):
    p = os.path.join("features", f)
    print(f"{f:30s} {os.path.getsize(p)/1e6:8.2f} MB")


## Notes pratiques

### W&B — setup rapide
```bash
pip install wandb python-dotenv
# Crée un fichier .env à côté du notebook :
# WANDB_API_KEY=xxxxxxxxxxxxxxxxxxxxxxxx
```
Ajoute `.env` à ton `.gitignore`.

Si tu veux désactiver W&B ponctuellement, passe `use_wandb=False` à `run_kfold`.

### Reprise d'entraînement
Si un fold crashe en cours, relance la cellule d'entraînement avec `resume=True` :
```python
convnext_path = run_kfold(..., resume=True)
```
La fonction recharge automatiquement le `_last.pth` du fold et reprend à l'epoch suivante.

### Checkpoints produits par fold
- `models/best_ConvNeXtBase_fold{N}.pth` — meilleur val_loss du fold
- `models/best_ConvNeXtBase_fold{N}_last.pth` — dernier epoch du fold (pour resume)
- `models/best_GLOBAL_ConvNeXtBase.pth` — copie du meilleur fold, utilisé à l'éval

### Si OOM sur RTX 5070 12 GB
Par ordre de préférence :
1. `ConvNeXtMetricLearning(grad_checkpointing=True)` (-35% VRAM, +25% temps)
2. `batch_size=8, accum_steps=3` (batch effectif 24 conservé)
3. `img_size=320` pour ConvNeXt
4. Remplacer ConvNeXt-Base par ConvNeXt-Small (`convnext_small.fb_in22k_ft_in1k`)
